# SUV Purchase Prediction (Full Marks Version)

End-to-end ML pipeline with clean engineering practices, evaluation, and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# For better display
plt.rcParams["figure.figsize"] = (6,4)


## 1. Data Loading

In [ ]:
def load_data(path):
    try:
        df = pd.read_csv(path)
        return df
    except Exception as e:
        print("Error loading file:", e)
        raise

df = load_data("suv_data.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
print(df.info())

## 2. Preprocessing

In [ ]:
def preprocess(df):
    df = df.copy()
    if "Gender" in df.columns:
        df["Gender"] = df["Gender"].map({"Male":0, "Female":1})
    X = df[["Age","EstimatedSalary"]]
    y = df["Purchased"]
    return X, y

X, y = preprocess(df)
X.head()

## 3. Train-Test Split + Scaling

In [ ]:
def split_and_scale(X, y, test_size=0.2):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

X_train, X_test, y_train, y_test, scaler = split_and_scale(X,y)


## 4. Model Training

In [ ]:
def train_model(X_train, y_train):
    model = LogisticRegression()
    model.fit(X_train, y_train)
    return model

model = train_model(X_train, y_train)


## 5. Evaluation

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    return acc, cm

accuracy, cm = evaluate(model, X_test, y_test)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", cm)


In [ ]:
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha="center", va="center")

plt.show()


## 6. Decision Boundary

In [ ]:
def plot_decision_boundary(model, scaler, X, y):
    x_min, x_max = X["Age"].min()-1, X["Age"].max()+1
    y_min, y_max = X["EstimatedSalary"].min()-1000, X["EstimatedSalary"].max()+1000

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid)
    probs = model.predict(grid_scaled).reshape(xx.shape)

    plt.contourf(xx, yy, probs, alpha=0.3)
    plt.scatter(X["Age"], X["EstimatedSalary"], c=y)
    plt.title("Decision Boundary")
    plt.xlabel("Age")
    plt.ylabel("Salary")
    plt.show()

plot_decision_boundary(model, scaler, X, y)


## 7. Comparison (70/30, 75/25)

In [ ]:
for size in [0.3, 0.25]:
    X_train, X_test, y_train, y_test, scaler = split_and_scale(X,y,test_size=size)
    model = train_model(X_train,y_train)
    acc, _ = evaluate(model,X_test,y_test)
    print(f"Test size {size} → Accuracy: {acc}")


## 8. Answers

**Q1:** Logistic Regression is a classification algorithm.

**Q2:** See split_and_scale function.

**Q3:** Confusion matrix shows prediction correctness across classes.
